# Gene Predictability Analysis v2 — with Data/Tile Diagnostics

**Hypothesis:** Per-gene test performance is largely determined by intrinsic gene properties, not by model capacity limitations.

**Analyses:**
1. Data size per gene (train/val/test coverage)
2. Per-tile test performance (are some tiles easier?)
3. Performance vs gene property correlations
4. Top vs bottom gene comparison
5. Random Forest: do gene properties jointly predict performance?
6. Tier-based summary

**Goal:** Prove that low-r genes reflect intrinsic unpredictability, not model failure.

## 1. Setup

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score

plt.rcParams.update({
    'figure.dpi':           120,
    'font.size':             10,
    'axes.spines.top':       False,
    'axes.spines.right':     False,
    'legend.frameon':        False,
})

PROJECT = Path('/hpc/group/jilab/tc459/MorphPT')

# Main LoRA results
EXP_DIR = PROJECT / 'experiments' / 'lora_probing_crc_multi_10.0x_mse'
RESULTS_FILE = EXP_DIR / 'multi_lora_hybrid_results.csv'
SPLITS_FILE = EXP_DIR / 'splits_seed_42.npz'

# Gene statistics
GENE_STATS_FILE = (PROJECT / 'cache_crc' / 'per_gene' /
                   'top400_variance_mincov0.1.csv')

# Cache paths
CACHE_DIR = PROJECT / 'cache_crc'
META_FILE = CACHE_DIR / 'meta.csv'
EXPR_FILE = CACHE_DIR / 'expr.npy'

print(f'Files to load:')
print(f'  Results:  {RESULTS_FILE}')
print(f'  Splits:   {SPLITS_FILE}')
print(f'  Stats:    {GENE_STATS_FILE}')
print(f'  Meta:     {META_FILE}')
print(f'  Expr:     {EXPR_FILE}')

Files to load:
  Results:  /hpc/group/jilab/tc459/MorphPT/experiments/lora_probing_crc_multi_10.0x_mse/multi_lora_hybrid_results.csv
  Splits:   /hpc/group/jilab/tc459/MorphPT/experiments/lora_probing_crc_multi_10.0x_mse/splits_seed_42.npz
  Stats:    /hpc/group/jilab/tc459/MorphPT/cache_crc/per_gene/top400_variance_mincov0.1.csv
  Meta:     /hpc/group/jilab/tc459/MorphPT/cache_crc/meta.csv
  Expr:     /hpc/group/jilab/tc459/MorphPT/cache_crc/expr.npy


## 2. Load data

In [2]:
# Load LoRA multi-task results
df_results = pd.read_csv(RESULTS_FILE)

# Load gene stats
df_stats = pd.read_csv(GENE_STATS_FILE)

# Merge
df = df_results.merge(df_stats, on='gene_name', suffixes=('_res', '_stats'))

# Rename test performance column
if 'test_pearson_s42' in df.columns:
    df['test_r'] = df['test_pearson_s42']
elif 'test_pearson_mean' in df.columns:
    df['test_r'] = df['test_pearson_mean']

# Compute derived features
if 'mean_expr' in df.columns and 'std_expr' in df.columns:
    df['cv']  = df['std_expr'] / (df['mean_expr'] + 0.001)
    df['snr'] = df['mean_expr'] / (df['std_expr'] + 0.001)

print(f'Merged: {df.shape}')
print(f'Available columns: {df.columns.tolist()}')
df.head(3)

Merged: (400, 11)
Available columns: ['gene_idx_res', 'gene_name', 'val_pearson_s42', 'test_pearson_s42', 'test_pearson_mean', 'test_pearson_std', 'gene_idx_stats', 'variance', 'coverage', 'rank', 'test_r']


,gene_idx_res,gene_name,val_pearson_s42,test_pearson_s42,test_pearson_mean,test_pearson_std,gene_idx_stats,variance,coverage,rank,test_r
0,269,IGKC,0.759872,0.732030,0.732030,NaN,269,4.926481,0.384976,1,0.732030
1,315,COL3A1,0.823657,0.779926,0.779926,NaN,315,3.189419,0.257538,2,0.779926
2,1945,CEACAM6,0.788160,0.769953,0.769953,NaN,1945,3.024286,0.683125,3,0.769953


## 3. Data size diagnostics

Per-gene train/val/test sample counts:
- Total cells
- Cells with **non-zero expression** per split (the actual "signal" samples)

Low-coverage genes have very few non-zero training samples — this is important context.

In [3]:
# Load splits and expression matrix
splits = np.load(SPLITS_FILE, allow_pickle=True)
train_idx = splits['train_idx']
val_idx   = splits['val_idx']
test_idx  = splits['test_idx']
test_tiles = splits['test_tiles']

print(f'Split sizes:')
print(f'  Train: {len(train_idx):>6,} cells')
print(f'  Val:   {len(val_idx):>6,} cells')
print(f'  Test:  {len(test_idx):>6,} cells')
print(f'  Total: {len(train_idx) + len(val_idx) + len(test_idx):>6,} cells')
print(f'\n  Test tiles: {test_tiles.tolist()}')

Split sizes:
  Train: 143,007 cells
  Val:   15,889 cells
  Test:  26,659 cells
  Total: 185,555 cells

  Test tiles: [17, 19, 10, 3]


In [4]:
# Load meta and expression matrix
meta = pd.read_csv(META_FILE)
all_meta = meta[meta['split'] != 'excluded'].reset_index(drop=True)

# Memory-map the expression matrix (shape: N x G)
expr = np.load(EXPR_FILE, mmap_mode='r')
print(f'Expression matrix shape: {expr.shape}')
print(f'Meta shape:              {all_meta.shape}')

# Sanity check alignment
assert expr.shape[0] == len(all_meta), \
    f'Expression rows ({expr.shape[0]}) != meta rows ({len(all_meta)})'
print('✓ Expression and meta aligned')

Expression matrix shape: (185555, 2220)
Meta shape:              (185555, 8)
✓ Expression and meta aligned


In [ ]:
# Subset expression to each split
# Use the gene indices from df
gene_indices = df['gene_idx_res'].values.astype(int)
print(f'Computing counts for {len(gene_indices)} genes...')

# For each gene, count non-zero cells per split
train_expr = expr[train_idx][:, gene_indices]
val_expr   = expr[val_idx][:, gene_indices]
test_expr  = expr[test_idx][:, gene_indices]

print(f'  Train expression subset: {train_expr.shape}')
print(f'  Val expression subset:   {val_expr.shape}')
print(f'  Test expression subset:  {test_expr.shape}')

# Count non-zero cells per gene
df['n_train_cells']     = len(train_idx)
df['n_val_cells']       = len(val_idx)
df['n_test_cells']      = len(test_idx)

df['train_nonzero']     = (train_expr > 0).sum(axis=0)
df['val_nonzero']       = (val_expr > 0).sum(axis=0)
df['test_nonzero']      = (test_expr > 0).sum(axis=0)

df['train_coverage']    = df['train_nonzero'] / df['n_train_cells']
df['val_coverage']      = df['val_nonzero']   / df['n_val_cells']
df['test_coverage']     = df['test_nonzero']  / df['n_test_cells']

print('\nDone. Added columns: train_nonzero, val_nonzero, test_nonzero, train_coverage, val_coverage, test_coverage')

Computing counts for 400 genes...


In [ ]:
# Summary of data sizes
print('Per-gene data size summary:\n')
print(f"{'':25s}{'mean':>12}{'median':>12}{'min':>12}{'max':>12}")
for col in ['train_nonzero', 'val_nonzero', 'test_nonzero']:
    s = df[col]
    print(f"{col:25s}{s.mean():>12.0f}{s.median():>12.0f}{s.min():>12.0f}{s.max():>12.0f}")

print(f"\nCoverage (fraction of cells expressing):\n")
print(f"{'':25s}{'mean':>12}{'median':>12}{'min':>12}{'max':>12}")
for col in ['train_coverage', 'val_coverage', 'test_coverage']:
    s = df[col]
    print(f"{col:25s}{s.mean():>12.3f}{s.median():>12.3f}{s.min():>12.3f}{s.max():>12.3f}")

In [ ]:
# Coverage consistency across splits (sanity check)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Train vs val
ax = axes[0]
ax.scatter(df['train_coverage'], df['val_coverage'], alpha=0.5, s=12)
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('Train coverage')
ax.set_ylabel('Val coverage')
r = pearsonr(df['train_coverage'], df['val_coverage'])[0]
ax.set_title(f'Train vs Val coverage\nr = {r:.3f}')
ax.grid(alpha=0.3)

# Train vs test
ax = axes[1]
ax.scatter(df['train_coverage'], df['test_coverage'], alpha=0.5, s=12)
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('Train coverage')
ax.set_ylabel('Test coverage')
r = pearsonr(df['train_coverage'], df['test_coverage'])[0]
ax.set_title(f'Train vs Test coverage\nr = {r:.3f}')
ax.grid(alpha=0.3)

# Val vs test
ax = axes[2]
ax.scatter(df['val_coverage'], df['test_coverage'], alpha=0.5, s=12)
ax.plot([0, 1], [0, 1], 'k--', lw=1, alpha=0.5)
ax.set_xlabel('Val coverage')
ax.set_ylabel('Test coverage')
r = pearsonr(df['val_coverage'], df['test_coverage'])[0]
ax.set_title(f'Val vs Test coverage\nr = {r:.3f}')
ax.grid(alpha=0.3)

fig.suptitle('Coverage consistency across splits (should be highly correlated)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Per-tile test performance analysis

Test set consists of 4 spatially held-out tiles. This analysis answers:
- Do different tiles have different **number of cells**?
- Do different tiles have different **cell type composition**?
- Do different genes predict **consistently across tiles** or **tile-specific**?

If per-tile r values vary dramatically for the same gene, it suggests tile-specific biology or test set composition effects.

In [ ]:
# Need per-cell predictions to compute per-tile r
# Check if saved predictions exist
pred_files_exist = {
    'y_true': (EXP_DIR / 'test_y_true_seed_42.npy').exists(),
    'y_pred': (EXP_DIR / 'test_y_pred_seed_42.npy').exists(),
}

print('Saved prediction files:')
for name, exists in pred_files_exist.items():
    status = '✓' if exists else '✗'
    print(f'  {status} {name}: {EXP_DIR / f"test_{name}_seed_42.npy"}')

HAS_PREDICTIONS = all(pred_files_exist.values())
print(f'\nPer-tile analysis possible: {HAS_PREDICTIONS}')

In [ ]:
# Test tile composition — without predictions, we can still show:
# 1. Cells per tile
# 2. Gene coverage per tile

test_meta = all_meta.iloc[test_idx].reset_index(drop=True)

print(f'Test tiles: {test_tiles.tolist()}')
print(f'\nCells per test tile:\n')
tile_counts = test_meta['tile_id'].value_counts().sort_index()
for tile_id, count in tile_counts.items():
    pct = 100 * count / len(test_meta)
    print(f'  Tile {tile_id:>3}: {count:>6,} cells ({pct:.1f}%)')

print(f'\n  Total: {len(test_meta):>6,} cells')

In [ ]:
# Per-tile coverage for each gene
# This shows: does Tile 17 have more of gene X's expressing cells than Tile 19?

tile_coverages = {}

for tile_id in test_tiles:
    tile_mask_in_test = (test_meta['tile_id'] == tile_id).values
    tile_expr = test_expr[tile_mask_in_test]  # (n_tile_cells, 400)
    tile_nonzero = (tile_expr > 0).sum(axis=0)
    tile_coverage = tile_nonzero / max(1, tile_mask_in_test.sum())
    tile_coverages[int(tile_id)] = tile_coverage

# Add to df
for tile_id in test_tiles:
    df[f'test_tile_{tile_id}_coverage'] = tile_coverages[int(tile_id)]

# Summary: per-gene, how does coverage vary across tiles?
coverage_cols = [f'test_tile_{t}_coverage' for t in test_tiles]
df['tile_coverage_std'] = df[coverage_cols].std(axis=1)
df['tile_coverage_mean'] = df[coverage_cols].mean(axis=1)
df['tile_coverage_cv'] = df['tile_coverage_std'] / (df['tile_coverage_mean'] + 0.001)

print('Per-gene coverage variation across test tiles:\n')
print(df[['gene_name', 'test_r'] + coverage_cols +
        ['tile_coverage_std', 'tile_coverage_cv']].head(10).to_string(index=False))

In [ ]:
# Does tile-coverage variance correlate with test_r?
# If YES → genes with tile-inconsistent coverage predict worse
# If NO → performance is not driven by split luck

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Tile coverage std vs test r
ax = axes[0]
ax.scatter(df['tile_coverage_std'], df['test_r'], alpha=0.5, s=15, c=df['test_r'], cmap='viridis')
r = pearsonr(df['tile_coverage_std'], df['test_r'])[0]
ax.set_xlabel('Tile coverage std')
ax.set_ylabel('Test Pearson r')
ax.set_title(f'Tile coverage variation vs test r\nPearson = {r:.3f}')
ax.grid(alpha=0.3)

# Tile coverage CV vs test r
ax = axes[1]
ax.scatter(df['tile_coverage_cv'], df['test_r'], alpha=0.5, s=15, c=df['test_r'], cmap='viridis')
r = pearsonr(df['tile_coverage_cv'], df['test_r'])[0]
ax.set_xlabel('Tile coverage CV (std/mean)')
ax.set_ylabel('Test Pearson r')
ax.set_title(f'Tile coverage CV vs test r\nPearson = {r:.3f}')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Per-tile PEARSON for each gene
# This requires the saved predictions

per_tile_r = {}
if HAS_PREDICTIONS:
    y_true = np.load(EXP_DIR / 'test_y_true_seed_42.npy')
    y_pred = np.load(EXP_DIR / 'test_y_pred_seed_42.npy')
    print(f'Loaded predictions: y_true={y_true.shape}, y_pred={y_pred.shape}')

    # Compute per-tile r
    for tile_id in test_tiles:
        tile_mask = (test_meta['tile_id'] == tile_id).values
        yt = y_true[tile_mask]
        yp = y_pred[tile_mask]

        rs = []
        for g in range(yt.shape[1]):
            if yt[:, g].std() < 1e-6:
                rs.append(0.0)
            else:
                r, _ = pearsonr(yt[:, g], yp[:, g])
                rs.append(r if not np.isnan(r) else 0.0)
        per_tile_r[int(tile_id)] = np.array(rs)
        df[f'test_tile_{tile_id}_r'] = rs

    print(f'\nPer-tile test r computed for {len(test_tiles)} tiles')
    print(f'Added columns: ' + ', '.join([f"test_tile_{t}_r" for t in test_tiles]))
else:
    print('Skipping per-tile r computation (predictions not saved)')
    print('To get this analysis: re-run training with --save_predictions flag')

In [ ]:
# Per-tile r distribution analysis

if HAS_PREDICTIONS:
    r_cols = [f'test_tile_{t}_r' for t in test_tiles]
    df['tile_r_std']  = df[r_cols].std(axis=1)
    df['tile_r_mean'] = df[r_cols].mean(axis=1)

    print('Per-tile r summary:\n')
    for t in test_tiles:
        rs = df[f'test_tile_{t}_r']
        print(f'  Tile {t:>3}: mean r = {rs.mean():.4f}, median = {rs.median():.4f}, n_cells = {(test_meta.tile_id == t).sum()}')

    print(f'\nOverall test r (all 4 tiles combined): {df["test_r"].mean():.4f}')
    print(f'Mean of per-tile means:                  {df["tile_r_mean"].mean():.4f}')
    print(f'\nPer-gene r std across tiles:')
    print(f'  Mean:   {df["tile_r_std"].mean():.4f}')
    print(f'  Median: {df["tile_r_std"].median():.4f}')
    print(f'  Max:    {df["tile_r_std"].max():.4f}')

In [ ]:
# Visualize per-tile test r distributions
if HAS_PREDICTIONS:
    r_cols = [f'test_tile_{t}_r' for t in test_tiles]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Boxplot: per-tile r distributions
    ax = axes[0]
    data = [df[col].values for col in r_cols]
    labels = [f'Tile {t}\n(n={int((test_meta.tile_id == t).sum())})' for t in test_tiles]
    bp = ax.boxplot(data, labels=labels, patch_artist=True, showmeans=True,
                    meanprops={'marker': 'D', 'markerfacecolor': 'red', 'markersize': 6})
    for patch, color in zip(bp['boxes'], ['#2E86AB', '#A23B72', '#F0B441', '#2d6a4f']):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel('Per-gene Pearson r')
    ax.set_title('Distribution of test r across tiles')
    ax.axhline(df['test_r'].mean(), color='red', ls='--', alpha=0.5,
               label=f'Overall mean ({df["test_r"].mean():.3f})')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    # Tile r std distribution
    ax = axes[1]
    ax.hist(df['tile_r_std'], bins=40, color='#2E86AB', alpha=0.7, edgecolor='black')
    ax.axvline(df['tile_r_std'].median(), color='red', ls='--',
               label=f'Median = {df["tile_r_std"].median():.3f}')
    ax.set_xlabel('Per-gene r std across 4 tiles')
    ax.set_ylabel('Count')
    ax.set_title('How much does gene r vary between tiles?')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.show()

In [ ]:
# Most tile-inconsistent genes (high std across tiles)
if HAS_PREDICTIONS:
    print('Top 10 genes with MOST tile-variable performance:\n')
    r_cols = [f'test_tile_{t}_r' for t in test_tiles]
    cols_show = ['gene_name', 'test_r', 'tile_r_std'] + r_cols
    most_variable = df.nlargest(10, 'tile_r_std')[cols_show]
    print(most_variable.round(3).to_string(index=False))

    print('\n\nTop 10 genes with MOST consistent (lowest std) tile performance:\n')
    most_consistent = df.nsmallest(10, 'tile_r_std')[cols_show]
    print(most_consistent.round(3).to_string(index=False))

## 5. Univariate analysis — each gene property vs test R

In [ ]:
PROPERTIES_TO_EXAMINE = [
    ('std_expr',      'Expression std'),
    ('mean_expr',     'Mean expression'),
    ('train_coverage','Train coverage'),
    ('train_nonzero', 'Train non-zero count'),
    ('cv',            'CV (std/mean)'),
]

PROPERTIES_TO_EXAMINE = [
    (c, label) for (c, label) in PROPERTIES_TO_EXAMINE
    if c in df.columns
]

n_props = len(PROPERTIES_TO_EXAMINE)
n_cols  = 2
n_rows  = (n_props + 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(13, 4*n_rows))
axes = axes.flatten() if n_props > 1 else [axes]

correlations = []
for i, (col, label) in enumerate(PROPERTIES_TO_EXAMINE):
    ax = axes[i]
    x = df[col].values
    y = df['test_r'].values
    m = ~(np.isnan(x) | np.isnan(y))
    x, y = x[m], y[m]

    ax.scatter(x, y, c=y, cmap='viridis', alpha=0.6, s=15)
    p_r, p_p = pearsonr(x, y)
    s_r, _ = spearmanr(x, y)

    coef = np.polyfit(x, y, 1)
    x_sorted = np.sort(x)
    ax.plot(x_sorted, np.polyval(coef, x_sorted), 'r-', lw=1, alpha=0.8)
    ax.set_xlabel(label)
    ax.set_ylabel('Test Pearson r')
    ax.set_title(f'{label}\nPearson={p_r:.3f} (p={p_p:.1e})  Spearman={s_r:.3f}')
    ax.grid(alpha=0.3)

    correlations.append({'property': col, 'label': label, 'pearson_r': p_r, 'pearson_p': p_p, 'spearman_r': s_r})

for i in range(n_props, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

corr_df = pd.DataFrame(correlations)
print('\nCorrelation summary:')
print(corr_df[['label', 'pearson_r', 'spearman_r']].round(4).to_string(index=False))

## 6. Top vs bottom genes — biological profile

In [ ]:
df_sorted = df.sort_values('test_r', ascending=False).reset_index(drop=True)

cols_show = ['gene_name', 'test_r', 'std_expr', 'mean_expr', 'train_coverage', 'train_nonzero']
cols_show = [c for c in cols_show if c in df.columns]

print('='*80)
print('TOP 20 BEST-PREDICTED GENES')
print('='*80)
print(df_sorted.head(20)[cols_show].round(3).to_string(index=False))

print('\n' + '='*80)
print('BOTTOM 20 WORST-PREDICTED GENES')
print('='*80)
print(df_sorted.tail(20)[cols_show].round(3).to_string(index=False))

In [ ]:
top50    = df_sorted.head(50)
middle50 = df_sorted.iloc[175:225]
bottom50 = df_sorted.tail(50)

print('Comparative properties (top 50 vs middle 50 vs bottom 50):\n')
compare_cols = ['test_r', 'std_expr', 'mean_expr', 'train_coverage', 'train_nonzero']
compare_cols = [c for c in compare_cols if c in df.columns]

for col in compare_cols:
    print(f'  {col}:')
    print(f'    Top 50:     mean={top50[col].mean():>10.4f}  median={top50[col].median():>10.4f}')
    print(f'    Middle 50:  mean={middle50[col].mean():>10.4f}  median={middle50[col].median():>10.4f}')
    print(f'    Bottom 50:  mean={bottom50[col].mean():>10.4f}  median={bottom50[col].median():>10.4f}')
    print()

In [ ]:
# Visual comparison
plot_props = [c for c in ['std_expr', 'train_coverage', 'train_nonzero', 'mean_expr'] if c in df.columns][:4]

fig, axes = plt.subplots(1, len(plot_props), figsize=(4*len(plot_props), 5), sharey=False)
if len(plot_props) == 1:
    axes = [axes]

groups = [('Top 50', top50, '#2d6a4f'),
          ('Middle 50', middle50, '#F0B441'),
          ('Bottom 50', bottom50, '#C44E52')]

for ax, col in zip(axes, plot_props):
    positions = np.arange(len(groups))
    data = [g[1][col].values for g in groups]
    labels = [g[0] for g in groups]
    colors = [g[2] for g in groups]

    bp = ax.boxplot(data, positions=positions, patch_artist=True, widths=0.6,
                    showmeans=True, meanprops={'marker': 'D', 'markerfacecolor': 'red', 'markersize': 5})
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_xticks(positions)
    ax.set_xticklabels(labels)
    ax.set_ylabel(col)
    ax.set_title(col)
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('Gene properties differ systematically by predictability', fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Random Forest — do gene properties jointly predict performance?

If a simple model can predict test_r from gene properties alone, performance IS determined by gene properties — supporting the claim that low-r genes are intrinsically less predictable.

In [ ]:
feature_cols = [c for c in ['std_expr', 'mean_expr', 'train_coverage',
                            'train_nonzero', 'cv', 'tile_coverage_std']
                if c in df.columns]

print(f'Using features: {feature_cols}')

mask = df[feature_cols + ['test_r']].notna().all(axis=1)
X = df.loc[mask, feature_cols].values
y = df.loc[mask, 'test_r'].values

print(f'Samples: {len(X)} genes × {X.shape[1]} features')

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
cv_scores = cross_val_score(rf, X, y, cv=5, scoring='r2')
print(f'\n5-fold CV R²:')
print(f'  Folds:  {[f"{s:.3f}" for s in cv_scores]}')
print(f'  Mean:   {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')

rf.fit(X, y)
print(f'\nIn-sample R²: {rf.score(X, y):.3f}')

In [ ]:
# Feature importance
result = permutation_importance(rf, X, y, n_repeats=30, random_state=42, n_jobs=-1)

imp_df = pd.DataFrame({
    'feature':         feature_cols,
    'importance_mean': result.importances_mean,
    'importance_std':  result.importances_std,
}).sort_values('importance_mean', ascending=False)

print('Permutation importance:')
print(imp_df.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(imp_df['feature'][::-1], imp_df['importance_mean'][::-1],
        xerr=imp_df['importance_std'][::-1],
        color='#2E86AB', alpha=0.8, edgecolor='black', lw=0.5)
ax.set_xlabel('Permutation importance')
ax.set_title('Which gene properties predict test R?')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print('='*72)
print(' SUMMARY: Is performance determined by gene properties?')
print('='*72)

print(f'\n1. DATA SIZE (train/val/test):')
print(f'   Total train: {len(train_idx):,}  val: {len(val_idx):,}  test: {len(test_idx):,}')
print(f'   Mean train non-zero per gene: {df["train_nonzero"].mean():.0f}')
print(f'   Range: {df["train_nonzero"].min():.0f} - {df["train_nonzero"].max():.0f}')

print(f'\n2. UNIVARIATE CORRELATIONS:\n')
for corr in correlations:
    strength = ('STRONG'   if abs(corr['pearson_r']) > 0.5
                else 'MODERATE' if abs(corr['pearson_r']) > 0.3
                else 'WEAK')
    print(f'   {corr["label"]:<30}: r = {corr["pearson_r"]:+.3f}  [{strength}]')

print(f'\n3. JOINT PREDICTION (Random Forest):')
print(f'   5-fold CV R²: {cv_scores.mean():.3f} +/- {cv_scores.std():.3f}')

if cv_scores.mean() > 0.5:
    conclusion = 'STRONG EVIDENCE'
elif cv_scores.mean() > 0.3:
    conclusion = 'MODERATE EVIDENCE'
else:
    conclusion = 'WEAK EVIDENCE'

print(f'   -> {conclusion}: gene properties determine predictability')

if HAS_PREDICTIONS:
    print(f'\n4. PER-TILE CONSISTENCY:')
    for t in test_tiles:
        rs = df[f'test_tile_{t}_r']
        print(f'   Tile {t:>3}: mean r = {rs.mean():.4f}')
    print(f'   Per-gene r std (across tiles): {df["tile_r_std"].mean():.4f}')

# Save
out_path = PROJECT / 'experiments' / 'gene_predictability_analysis_v2.csv'
df.to_csv(out_path, index=False)
print(f'\nSaved to: {out_path}')